In [1]:
# STEP 1 — INSTALL REQUIRED LIBRARIES

import sys

!{sys.executable} -m pip install llama-index
!{sys.executable} -m pip install llama-index-llms-openai
!{sys.executable} -m pip install pypdf
!{sys.executable} -m pip install openai==0.28.1
!{sys.executable} -m pip install python-dotenv
!{sys.executable} -m pip install nest_asyncio

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 1.8 MB/s eta 0:00:00a 0:00:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-agent-openai 0.2.3 requires openai>=1.14.0, but you have openai 0.28.1 which is incompatible.
llama-index-core 0.10.30 requires openai>=1.1.0, but you have openai 0.28.1 which is incompatible.
llama-index-legacy 0.9.48 requires op

In [2]:
# STEP 2 — IMPORT REQUIRED LIBRARIES

import os
import openai
import nest_asyncio
import urllib.request

from dotenv import load_dotenv, find_dotenv

nest_asyncio.apply()

In [3]:
# STEP 3 — LOAD API KEY

_ = load_dotenv(find_dotenv())

openai.api_key = os.environ['OPENAI_API_KEY']

print("API Loaded Successfully")

API Loaded Successfully


In [4]:
# STEP 4 — SELECT RESEARCH PAPERS

urls = [
    "https://openreview.net/pdf?id=LzPWWPAdY4",
    "https://openreview.net/pdf?id=VTF8yNQM66",
    "https://openreview.net/pdf?id=yV6fD7LYkF",
    "https://openreview.net/pdf?id=hnrB5YHoYu",
    "https://openreview.net/pdf?id=WbWtOYIzIK"
]

papers = [
    "loftq.pdf",
    "swebench.pdf",
    "values.pdf",
    "finetune_fair_diffusion.pdf",
    "knowledge_card.pdf"
]

In [5]:
# STEP 5 — DOWNLOAD PAPERS

for url, paper in zip(urls, papers):

    print(f"Downloading {paper}...")

    urllib.request.urlretrieve(url, paper)

print("Download Completed")

Download Completed


In [6]:
# STEP 6 — VERIFY DOWNLOADED FILES

import os

print(os.listdir())

['L4_Building_a_Multi-Document_Agent.ipynb', 'finetune_fair_diffusion.pdf', 'helper.py', 'knowledge_card.pdf', 'loftq.pdf', 'longlora.pdf', 'metagpt.pdf', 'metra.pdf', 'selfrag.pdf', 'swebench.pdf', 'utils.py', 'values.pdf', 'vr_mcl.pdf', 'zipformer.pdf', '.ipynb_checkpoints', 'Untitled.ipynb']


In [8]:
from pathlib import Path

from llama_index.core import VectorStoreIndex
from llama_index.core.tools import QueryEngineTool
from llama_index.core import SimpleDirectoryReader
from llama_index.core.objects import ObjectIndex

from llama_index.llms.openai import OpenAI

from llama_index.core.agent import FunctionCallingAgentWorker
from llama_index.core.agent import AgentRunner

In [9]:
paper_to_tools_dict = {}

for paper in papers:

    print(f"Processing {paper}")

    documents = SimpleDirectoryReader(
        input_files=[paper]
    ).load_data()

    index = VectorStoreIndex.from_documents(documents)

    query_engine = index.as_query_engine()

    summary_tool = QueryEngineTool.from_defaults(
        query_engine=query_engine,
        name=f"{Path(paper).stem}_summary",
        description=f"Provides summary of {paper}"
    )

    vector_tool = QueryEngineTool.from_defaults(
        query_engine=query_engine,
        name=f"{Path(paper).stem}_vector",
        description=f"Answers detailed questions from {paper}"
    )

    paper_to_tools_dict[paper] = [
        vector_tool,
        summary_tool
    ]

Processing loftq.pdf
Processing swebench.pdf
Processing values.pdf
Processing finetune_fair_diffusion.pdf
Processing knowledge_card.pdf


In [10]:
all_tools = [
    t for paper in papers
    for t in paper_to_tools_dict[paper]
]

print(len(all_tools))

10


In [12]:
obj_index = ObjectIndex.from_objects(
    all_tools,
    index_cls=VectorStoreIndex,
)

obj_retriever = obj_index.as_retriever(
    similarity_top_k=3
)
llm = OpenAI(
    model="gpt-3.5-turbo"
)

In [14]:

agent_worker = FunctionCallingAgentWorker.from_tools(

    tool_retriever=obj_retriever,

    llm=llm,

    system_prompt="""
You are an agent designed to answer queries
over multiple research papers.

Always use the provided tools.

Do not rely on prior knowledge.
""",

    verbose=True
)

agent = AgentRunner(agent_worker)

In [15]:
response = agent.query(
    "What problem does LoftQ solve?"
)

print(response)

Added user message to memory: What problem does LoftQ solve?
=== Calling Function ===
Calling function: loftq_summary with args: {"input": "problem"}
=== Function Output ===
The problem discussed in the provided context is related to various research papers and challenges in the field of natural language processing, machine learning, and neural networks.
=== LLM Response ===
LoftQ addresses challenges and research topics in natural language processing, machine learning, and neural networks.
assistant: LoftQ addresses challenges and research topics in natural language processing, machine learning, and neural networks.


In [19]:
 
response = agent.query(
    "Compare SWEBench and LoftQ papers"
)
print("Narendran K - 212223230135")
print(response)

Added user message to memory: Compare SWEBench and LoftQ papers
=== Calling Function ===
Calling function: loftq_summary with args: {"input": "summary"}
=== Function Output ===
The provided context information includes references to various papers and works related to natural language processing, machine learning, and neural networks. It mentions works such as "Attention is all you need," "GLUE: A multi-task benchmark," "BART: Denoising sequence-to-sequence pre-training," "ROUGE: A package for automatic evaluation of summaries," and others.
=== Calling Function ===
Calling function: swebench_vector with args: {"input": "summary"}
=== Function Output ===
The research paper discusses the importance of ensuring AI-generated code aligns with human intents to address potential issues in machine-automated software engineering. The paper introduces SWE-bench as a testbed for designing safe and robust measures in AI-driven software engineering. Additionally, the paper provides qualitative anal

In [18]:
response = agent.query(
    "Which paper discusses fairness in AI systems?"
)

print(response)

Added user message to memory: Which paper discusses fairness in AI systems?
=== Calling Function ===
Calling function: finetune_fair_diffusion_summary with args: {"input": "fairness in AI systems"}
=== Function Output ===
Fairness in AI systems is a critical concern that needs to be addressed to prevent biases from propagating and limiting opportunities for minority groups. Approaches like distributional alignment loss and adjusted direct finetuning of diffusion models can help reduce biases related to gender, race, and other attributes in generated images. It is important to consider diverse perspectives of fairness beyond absolute equality and ensure that AI systems are ethically developed to provide a more balanced representation of different protected groups.
=== LLM Response ===
The paper that discusses fairness in AI systems highlights the importance of addressing biases to prevent discrimination and ensure equal opportunities for all groups. It suggests approaches like distribut